# Simple RAG Application with LangGraph

A simple sequential RAG system using LangGraph for leave policy questions.

## 1. Install Required Libraries

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community
!pip install -q chromadb
!pip install -q langchain-chroma
!pip install -q pypdf

## 2. Import Libraries

In [1]:
import os
from typing import TypedDict
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

## 3. Configuration

In [2]:
# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 4. Setup Vector Database

### 4.1 Initialize Text Splitter

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

print("Text splitter initialized!")

Text splitter initialized!


### 4.2 Load Leave Policy Vector Database

In [5]:
LEAVE_POLICY_PDF_PATH = "/home/mani/PatronusAI/agentic_rag_memory/Company_Leave_Policies_Extended.pdf"

def load_vectorstore(pdf_path):
    if not os.path.exists(pdf_path):
        print(f"Warning: PDF not found at {pdf_path}")
        return None
    
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    split_docs = text_splitter.split_documents(documents)
    
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        collection_name="leave_policies",
        persist_directory="./chroma_db_simple"
    )
    
    print(f"Loaded {len(documents)} pages, split into {len(split_docs)} chunks")
    return vectorstore

vectorstore = load_vectorstore(LEAVE_POLICY_PDF_PATH)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) if vectorstore else None

Loaded 7 pages, split into 7 chunks


## 5. Define Graph State

In [6]:
class State(TypedDict):
    question: str
    context: str
    answer: str

## 6. Define Graph Nodes

### 6.1 Retrieve Node

In [7]:
def retrieve(state: State) -> State:
    """Retrieve relevant documents from vector database"""
    question = state['question']
    
    if retriever is None:
        return {**state, "context": "No documents loaded"}
    
    # Retrieve documents
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return {**state, "context": context}

### 6.2 Generate Node

In [8]:
def generate(state: State) -> State:
    """Generate answer based on retrieved context"""
    question = state['question']
    context = state['context']
    
    # Create prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful HR assistant. Answer the question based on the provided leave policy context."),
        ("human", """Context:
{context}

Question: {question}

Provide a clear and concise answer based on the context above.""")
    ])
    
    # Generate response
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    
    return {**state, "answer": response.content}

## 7. Build Sequential RAG Graph

In [9]:
# Create graph
workflow = StateGraph(State)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

# Add edges (sequential flow)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile graph
app = workflow.compile()

print("Sequential RAG graph compiled!")

Sequential RAG graph compiled!


## 8. Query Function

In [10]:
def query_rag(question: str) -> str:
    """Query the RAG system
    
    Args:
        question: Question about leave policies
        
    Returns:
        Answer based on retrieved context
    """
    result = app.invoke({"question": question})
    return result['answer']

## 9. Example Usage

In [11]:
# Example 1: Annual leave question
answer = query_rag("How many days of annual leave are employees entitled to?")
print("Question: How many days of annual leave are employees entitled to?")
print(f"Answer: {answer}")
print("\n" + "="*80 + "\n")

Question: How many days of annual leave are employees entitled to?
Answer: Employees are entitled to the following annual leave based on their level:

- Junior employees: 18 days
- Mid-level employees: 22 days
- Senior employees: 28 days




In [12]:
# Example 2: Customized Question
answer = query_rag("How many days of annual leave do I get?")
print("Question: How many days of annual leave do I get?")
print(f"Answer: {answer}")
print("\n" + "="*80 + "\n")

Question: How many days of annual leave do I get?
Answer: The number of days of annual leave you get depends on your employment level:

- Junior: 18 days
- Mid-level: 22 days
- Senior: 28 days

Please refer to your specific employment level to determine your annual leave entitlement.


